# 🛒 Walmart Weekly Sales Forecasting using SARIMAX

This notebook builds a store-level weekly sales forecasting pipeline using the **Walmart Retail Dataset** and **SARIMAX** (Seasonal AutoRegressive Integrated Moving Average with eXogenous variables).

### Workflow:
1. Import Libraries & Load Dataset
2. Exploratory Data Analysis (EDA)
3. Store Selection & Weekly Sales Aggregation
4. Seasonal Decomposition
5. Store Comparison (2012)
6. SARIMAX Model Training
7. One-Step Ahead Forecast & MSE
8. Dynamic Forecast & RMSE
9. 12-Week Ahead Forecast
10. Conclusion

## 1. Import Libraries & Load Dataset

The Walmart dataset contains **6,435 weekly sales records** across **45 stores** with features including sales, CPI, unemployment rate, temperature, and fuel price.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

df = pd.read_csv('Walmart.csv').copy()
df.set_index('Date', inplace=True)

print('Dataset Shape :', df.shape)
print('Stores        :', df['Store'].nunique())
print('Columns       :', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

## 2. Exploratory Data Analysis (EDA)

Before modelling, we explore the dataset to understand:
- Overall sales distribution
- Top and bottom performing stores
- Impact of external factors (CPI, unemployment, temperature, fuel price) on weekly sales
- Holiday vs non-holiday sales comparison

In [ ]:
# Basic statistics
print('=== Dataset Statistics ===')
print(df.describe().round(2))

print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Top and bottom performing stores by average weekly sales
store_avg = df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

store_avg.head(10).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Stores by Avg Weekly Sales', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Store ID')
axes[0].set_ylabel('Avg Weekly Sales ($)')
axes[0].tick_params(axis='x', rotation=0)

store_avg.tail(10).plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Bottom 10 Stores by Avg Weekly Sales', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Store ID')
axes[1].set_ylabel('Avg Weekly Sales ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('store_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best performing store  : Store {store_avg.idxmax()} (${store_avg.max():,.0f} avg/week)')
print(f'Worst performing store : Store {store_avg.idxmin()} (${store_avg.min():,.0f} avg/week)')

In [ ]:
# Holiday vs Non-Holiday sales
holiday_sales     = df[df['Holiday_Flag'] == 1]['Weekly_Sales'].mean()
non_holiday_sales = df[df['Holiday_Flag'] == 0]['Weekly_Sales'].mean()

print(f'Average Weekly Sales on Holidays     : ${holiday_sales:,.2f}')
print(f'Average Weekly Sales on Non-Holidays : ${non_holiday_sales:,.2f}')
print(f'Holiday Sales Premium                : {((holiday_sales/non_holiday_sales)-1)*100:.1f}%')

In [ ]:
# Correlation between external factors and weekly sales
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

factors = [
    ('Temperature',  'tomato',    axes[0][0]),
    ('Fuel_Price',   'goldenrod', axes[0][1]),
    ('CPI',          'steelblue', axes[1][0]),
    ('Unemployment', 'mediumseagreen', axes[1][1]),
]

for col, color, ax in factors:
    ax.scatter(df[col], df['Weekly_Sales'], alpha=0.3, color=color, s=10)
    corr = df[col].corr(df['Weekly_Sales'])
    ax.set_title(f'{col} vs Weekly Sales  (r = {corr:.3f})', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Weekly Sales ($)')

plt.suptitle('Impact of External Factors on Weekly Sales', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('factor_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Store Selection & Weekly Sales Aggregation

The dataset contains 45 stores. We allow selection of any store (1–45) and aggregate its weekly sales for time series analysis.

In [ ]:
# Select store — change this value to analyse any store (1–45)
STORE_ID = 1

store  = df[df['Store'] == STORE_ID]
sales  = pd.DataFrame(store['Weekly_Sales'].groupby(store.index).sum())
sales.reset_index(inplace=True)
sales['Date'] = pd.to_datetime(sales['Date'], dayfirst=True)
sales.set_index('Date', inplace=True)

print(f'Store {STORE_ID} — Weekly Sales Records: {len(sales)}')
print(f'Date Range: {sales.index.min().date()} to {sales.index.max().date()}')
print(f'Average Weekly Sales: ${sales["Weekly_Sales"].mean():,.2f}')

# Plot
plt.figure(figsize=(12, 5))
sales['Weekly_Sales'].plot(color='steelblue', linewidth=1.2)
plt.title(f'Store {STORE_ID} — Weekly Sales Over Time', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Weekly Sales ($)')
plt.tight_layout()
plt.savefig(f'store{STORE_ID}_sales.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Seasonal Decomposition

We decompose the sales time series into **Trend**, **Seasonality**, and **Residuals** to understand the underlying patterns before building a model.

In [ ]:
decomposition = seasonal_decompose(sales['Weekly_Sales'], period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomposition.observed.plot(ax=axes[0],  color='steelblue')
axes[0].set_title('Observed',    fontweight='bold')
axes[0].set_ylabel('Sales ($)')

decomposition.trend.plot(ax=axes[1],     color='darkorange')
axes[1].set_title('Trend',       fontweight='bold')
axes[1].set_ylabel('Sales ($)')

decomposition.seasonal.plot(ax=axes[2],  color='green')
axes[2].set_title('Seasonality', fontweight='bold')
axes[2].set_ylabel('Sales ($)')

decomposition.resid.plot(ax=axes[3],     color='red')
axes[3].set_title('Residuals',   fontweight='bold')
axes[3].set_ylabel('Sales ($)')

plt.suptitle(f'Store {STORE_ID} — Seasonal Decomposition', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('seasonal_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Store Comparison — 2012

We compare the selected store's 2012 weekly sales against Store 5 to identify performance differences and seasonal patterns across stores.

In [ ]:
# Store 5 sales
store5        = df[df['Store'] == 5]
sales5        = pd.DataFrame(store5['Weekly_Sales'].groupby(store5.index).sum())
sales5.reset_index(inplace=True)
sales5['Date'] = pd.to_datetime(sales5['Date'], dayfirst=True)
sales5.set_index('Date', inplace=True)

y1 = sales['Weekly_Sales']
y2 = sales5['Weekly_Sales']

plt.figure(figsize=(14, 5))
y1['2012'].plot(label=f'Store {STORE_ID}', color='chocolate', linewidth=1.5)
y2['2012'].plot(label='Store 5',           color='turquoise', linewidth=1.5)
plt.ylabel('Weekly Sales ($)')
plt.xlabel('Date')
plt.title(f'Store {STORE_ID} vs Store 5 — Weekly Sales (2012)', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('store_comparison_2012.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Store {STORE_ID} avg sales in 2012 : ${y1['2012'].mean():,.2f}")
print(f"Store 5 avg sales in 2012  : ${y2['2012'].mean():,.2f}")

## 6. SARIMAX Model Training

We use **SARIMAX(4,4,3)(1,1,0,52)** — a seasonal ARIMA model with a yearly seasonal period (52 weeks).

- **(4,4,3)** — non-seasonal AR, differencing, MA orders
- **(1,1,0,52)** — seasonal AR, differencing, MA with 52-week seasonality

The model is fitted on the selected store's full historical sales data.

In [ ]:
model = sm.tsa.statespace.SARIMAX(
    y1,
    order=(4, 4, 3),
    seasonal_order=(1, 1, 0, 52),
    enforce_invertibility=False
)
results = model.fit(disp=False)

print('=== SARIMAX Model Summary ===')
print(results.summary().tables[1])

In [ ]:
# Model diagnostics — checks residual normality, autocorrelation, and heteroskedasticity
plt.style.use('seaborn-v0_8-pastel')
results.plot_diagnostics(figsize=(14, 10))
plt.suptitle('SARIMAX Model Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. One-Step Ahead Forecast & MSE

In **one-step ahead forecasting**, the model uses actual observed values at each step to predict the next one. This gives the best short-term accuracy and helps evaluate how well the model fits recent data.

In [ ]:
FORECAST_START = '2012-07-27'

pred    = results.get_prediction(start=pd.to_datetime(FORECAST_START), dynamic=False)
pred_ci = pred.conf_int()

plt.figure(figsize=(13, 6))
ax = plt.gca()
y1['2010':].plot(ax=ax, label='Observed', color='steelblue')
pred.predicted_mean.plot(ax=ax, label='One-Step Ahead Forecast', color='darkorange', alpha=0.8)
ax.fill_between(pred_ci.index, pred_ci.iloc[:, 0], pred_ci.iloc[:, 1], color='gray', alpha=0.2, label='Confidence Interval')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.set_title(f'Store {STORE_ID} — One-Step Ahead Forecast', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('one_step_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# MSE
y_forecasted = pred.predicted_mean
y_truth      = y1[FORECAST_START:]
mse          = ((y_forecasted - y_truth) ** 2).mean()
print(f'One-Step Ahead MSE : {round(mse, 2):,}')

## 8. Dynamic Forecast & RMSE

In **dynamic forecasting**, the model uses its own previous predictions (not actual values) to forecast forward. This simulates real future prediction where no actual data is available, and gives a more realistic measure of forecast accuracy.

In [ ]:
pred_dynamic    = results.get_prediction(start=pd.to_datetime(FORECAST_START), dynamic=True, full_results=True)
pred_dynamic_ci = pred_dynamic.conf_int()

plt.figure(figsize=(13, 6))
ax = plt.gca()
y1['2010':].plot(ax=ax, label='Observed', color='steelblue')
pred_dynamic.predicted_mean.plot(ax=ax, label='Dynamic Forecast', color='crimson')
ax.fill_between(pred_dynamic_ci.index, pred_dynamic_ci.iloc[:, 0], pred_dynamic_ci.iloc[:, 1], color='gray', alpha=0.25, label='Confidence Interval')
ax.fill_betweenx(ax.get_ylim(), pd.to_datetime(FORECAST_START), y1.index[-1], alpha=0.08, zorder=-1, color='yellow', label='Forecast Region')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.set_title(f'Store {STORE_ID} — Dynamic Forecast', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('dynamic_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# RMSE
y_forecasted_dyn = pred_dynamic.predicted_mean
y_truth_dyn      = y1[FORECAST_START:]
rmse             = np.sqrt(((y_forecasted_dyn - y_truth_dyn) ** 2).mean())
residual         = np.abs(y_forecasted_dyn - y_truth_dyn).sum()

print(f'Dynamic Forecast RMSE          : ${round(rmse, 2):,}')
print(f'Total Absolute Residual (Store {STORE_ID}): ${round(residual, 2):,}')

## 9. 12-Week Ahead Forecast

Finally, we forecast **12 weeks into the future** beyond the last available data point. This is the key business output — predicting upcoming weekly sales to support inventory planning and resource allocation.

In [ ]:
pred_uc = results.get_forecast(steps=12)
pred_ci = pred_uc.conf_int()

plt.figure(figsize=(13, 6))
ax = plt.gca()
y1.plot(ax=ax, label='Observed', color='steelblue')
pred_uc.predicted_mean.plot(ax=ax, label='12-Week Forecast', color='crimson', linewidth=2)
ax.fill_between(pred_ci.index, pred_ci.iloc[:, 0], pred_ci.iloc[:, 1], color='pink', alpha=0.4, label='95% Confidence Interval')
ax.axvline(y1.index[-1], color='gray', linestyle=':', linewidth=1.2, label='Forecast Start')
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.set_title(f'Store {STORE_ID} — 12-Week Sales Forecast', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('12week_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== 12-Week Forecast Summary ===')
forecast_df = pd.DataFrame({
    'Forecasted Sales ($)': pred_uc.predicted_mean.round(2),
    'Lower CI ($)':         pred_ci.iloc[:, 0].round(2),
    'Upper CI ($)':         pred_ci.iloc[:, 1].round(2)
})
print(forecast_df.to_string())

## 10. Conclusion

| Step | Detail |
|------|--------|
| Dataset | Walmart Retail Dataset — 6,435 records, 45 stores, 8 features |
| Features | Weekly Sales, CPI, Unemployment, Temperature, Fuel Price, Holiday Flag |
| Model | SARIMAX(4,4,3)(1,1,0,52) — yearly seasonal period |
| Forecast Types | One-step ahead (MSE) + Dynamic forecast (RMSE) |
| Business Output | 12-week ahead sales forecast with confidence intervals |

### Key Learnings
- External factors like CPI and unemployment influence weekly sales but the effect varies by store
- Holiday weeks consistently show higher sales — seasonal modelling must account for this
- One-step ahead forecasting is more accurate but requires actual data; dynamic forecasting is more realistic for real-world use
- Confidence intervals widen over the forecast horizon — uncertainty increases with time, which is expected and honest

---
**Author**: Karthikeyan K | B.Tech AI & Data Science | [GitHub](https://github.com/kt-keyan)